# Day 03: Deep Neural Networks — MNIST MLP

Welcome to practical 3! Build and train a multi-layer perceptron on a small MNIST subset.

This notebook follows the MIT lab style: abstract base classes, PyTorch, and LaTeX in markdown. Fill in methods marked `# YOUR CODE HERE`.

In [ ]:
import math
from abc import ABC, abstractmethod
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

device = torch.device("cpu")
torch.manual_seed(42)
np.random.seed(42)

We classify digits using a feed-forward network. Cross-entropy loss for one-hot labels $y$ and logits $z$ is $\mathcal{L} = -\sum_c y_c \log \mathrm{softmax}(z)_c$.

In [ ]:
from torchvision import datasets, transforms

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
mnist_train = datasets.MNIST(root="/tmp/mnist", train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(root="/tmp/mnist", train=False, download=True, transform=transform)
train_set = Subset(mnist_train, range(2000))
test_set = Subset(mnist_test, range(500))
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=256)

### Exercise 3.1: Define the MLP

**Your job**: Implement `MLP` with one hidden layer and ReLU activations.

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        # YOUR CODE HERE
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 10),
        )

    def forward(self, x):
        return self.net(x)

model = MLP().to(device)
print(model)

### Exercise 3.2: Training loop

**Your job**: Implement one epoch of SGD with cross-entropy loss.

In [ ]:
def train_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        # YOUR CODE HERE
        optimizer.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, correct / total

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []
for epoch in range(3):
    loss, acc = train_epoch(model, train_loader, optimizer)
    history.append((loss, acc))
    print(f"Epoch {epoch+1}: loss={loss:.4f}, acc={acc:.3f}")

### Exercise 3.3: Evaluation

**Your job**: Compute test accuracy.

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        # YOUR CODE HERE
        preds = model(x).argmax(1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    return correct / total

print("Test accuracy:", evaluate(model, test_loader))

### Exercise 3.4: Loss curve

**Your job**: Plot training loss per epoch.

In [ ]:
losses = [h[0] for h in history]
plt.plot(losses, marker="o")
plt.xlabel("epoch")
plt.ylabel("train loss")
plt.title("MNIST MLP training")
plt.show()

## Reflection (Day 3)

Take a few minutes to answer in your own words:

1. What was the most important concept you practiced today (deep neural networks and MNIST)?
2. Where did you get stuck, and how did you resolve it?
3. How would you explain today's material to a classmate in two sentences?
4. What would you like to explore further?